In [1]:
# --- Path plumbing (point to src) + autoreload ---
from pathlib import Path
import sys
import pandas as pd

%load_ext autoreload
%autoreload 2

NB   = Path.cwd()
ROOT = NB.parent
SRC  = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from data_pipeline.config import ROOT as ROOT_CFG, PROCESSED_DIR, CLEANED
print("CWD:", NB)
print("SRC:", SRC)
print("ROOT:", ROOT_CFG)
print("PROC:", PROCESSED_DIR)
print("CLEANED:", CLEANED)


# --- 0) Imports & paths ---
from pathlib import Path
import numpy as np

from simulator.env import HedgingEnv
from simulator.rewards import pnl_only
from rl_agent.policy_nn import PolicyNetwork, PolicyConfig
from rl_agent.trainer import train_reinforce, TrainConfig, evaluate_env

CWD: /Users/ya/Desktop/deep-hedging-rl/notebooks
SRC: /Users/ya/Desktop/deep-hedging-rl/src
ROOT: /Users/ya/Desktop/deep-hedging-rl
PROC: /Users/ya/Desktop/deep-hedging-rl/data/processed
CLEANED: /Users/ya/Desktop/deep-hedging-rl/data/processed/cleaned


In [2]:
# --- 1) Load panel exported from 02_simulator ---
panel = pd.read_csv(CLEANED / "hedging_panel_spx.csv", parse_dates=["date"])
panel = panel.sort_values("date").reset_index(drop=True)

print("Panel window:", panel["date"].min(), "→", panel["date"].max())
print("Columns:", list(panel.columns)[:12], "...")
assert "ret_fwd" in panel.columns, "CSV must include 'ret_fwd' computed in 02_simulator."


Panel window: 2005-01-21 00:00:00 → 2023-08-31 00:00:00
Columns: ['date', 'close_spy', 'vix', 'rate_10y', 'hvol_10d', 'hvol_14d', 'hvol_30d', 'hvol_60d', 'hvol_91d', 'hvol_122d', 'hvol_152d', 'hvol_182d'] ...


In [3]:
# --- 2) Define state columns (pick from panel columns that matter) ---
preferred = [
    "iv_atm_30d_spx", "iv_ts_slope_spx", "iv_skew_30d_spx",
    "vix", "rate_10y", "rv_21d", "hvol_30d", "hvol_91d",
]
extra_feats = ["ret_fwd", "close_spy"]  # or add a rolling z-score of returns

state_cols = [c for c in preferred if c in panel.columns]
if len(state_cols) < 4:
    extras = [c for c in panel.columns if c.startswith(("iv_", "skew", "term_slope")) and c not in state_cols]
    state_cols = list(dict.fromkeys(state_cols + extras[:3] + ["vix", "rate_10y", "rv_21d"]))
    state_cols = [c for c in state_cols if c in panel.columns]

state_cols = list(dict.fromkeys(state_cols + [c for c in extra_feats if c in panel.columns]))

print("STATE_COLS:", state_cols)
assert len(state_cols) > 0, "No usable state columns found in CSV."


STATE_COLS: ['iv_atm_30d_spx', 'iv_ts_slope_spx', 'iv_skew_30d_spx', 'vix', 'rate_10y', 'rv_21d', 'hvol_30d', 'hvol_91d', 'ret_fwd', 'close_spy']


In [4]:
# --- 4) Train/valid/test split + scaler (no leakage) ---
TRAIN_END = pd.Timestamp("2016-12-31")
VALID_END = pd.Timestamp("2019-12-31")

mask_train = panel["date"] <= TRAIN_END
mask_valid = (panel["date"] > TRAIN_END) & (panel["date"] <= VALID_END)
mask_test  = panel["date"] > VALID_END

mu = panel.loc[mask_train, state_cols].mean()
sigma = panel.loc[mask_train, state_cols].std(ddof=1).replace(0, np.nan).fillna(1.0)
scaler = lambda obs: (obs - mu.values) / sigma.values

print("Rows (train/valid/test):", mask_train.sum(), mask_valid.sum(), mask_test.sum())


Rows (train/valid/test): 7086 3273 7659


In [5]:
# --- 5) Build environments (NaN-safe, bps scaling, pos_limit) ---
from rl_agent.trainer import reward_bps

env_train = HedgingEnv(
    df=panel.loc[mask_train].reset_index(drop=True),
    features=state_cols,
    reward_fn=reward_bps,
    window=45,
    txn_cost_bps=1.0,
    scaler=scaler,
    hold_on_nan=True,
    pos_limit=2.0,
)

env_valid = HedgingEnv(
    df=panel.loc[mask_valid].reset_index(drop=True),
    features=state_cols,
    reward_fn=reward_bps,
    window=45,
    txn_cost_bps=1.0,
    scaler=scaler,
    hold_on_nan=True,
    pos_limit=2.0,
)

env_test = HedgingEnv(
    df=panel.loc[mask_test].reset_index(drop=True),
    features=state_cols,
    reward_fn=reward_bps,
    window=45,
    txn_cost_bps=1.0,
    scaler=scaler,
    hold_on_nan=True,
    pos_limit=2.0,
)

obs0 = env_train.reset()
input_dim = obs0.size
print("Env(train) ready. Flattened input_dim:", input_dim)


Env(train) ready. Flattened input_dim: 450


In [6]:
# --- 6) Quick baselines to confirm pipeline sanity ---
def always_long(_obs): return 1.0

res_al = env_train.rollout(lambda _: 1.0)
print("Always long Sharpe:", np.mean(res_al["rewards"]) / np.std(res_al["rewards"]) * np.sqrt(252))

try:
    from simulator.baselines import volatility_targeting
    feat_idx = state_cols.index("vix") if "vix" in state_cols else 0
    res_vt = env_train.rollout(volatility_targeting(feature_idx=feat_idx, ann_vol_target=0.15))
    rvt = np.asarray(res_vt["rewards"], float)
    sr_vt = (rvt.mean()/rvt.std(ddof=1)*np.sqrt(252)) if rvt.std(ddof=1) > 0 else 0.0
    print("Vol-target Sharpe (train):", f"{sr_vt:.3f}")
except Exception as e:
    print("Vol-target baseline skipped:", e)


Always long Sharpe: 0.3466778309147336
Vol-target Sharpe (train): 0.270


In [7]:
# --- 7) Initialize policy & train (REINFORCE) ---
policy = PolicyNetwork(PolicyConfig(input_dim=input_dim, hidden=128, init_log_std=-0.3, device="cpu"))

cfg = TrainConfig(
    gamma=0.99,
    entropy_coef=1e-2,  # can anneal to 0.0 later
    lr=1e-3,
    max_steps=None,
    device="cpu",
    print_every=50
)

# train on TRAIN only
history = train_reinforce(env_train, policy, cfg)

# evaluate deterministically
from rl_agent.trainer import evaluate_env
print("VALID eval:", evaluate_env(env_valid, policy, deterministic=True))
print("TEST  eval:", evaluate_env(env_test,  policy, deterministic=True))


[EP 0050] R:-74.9371  T:7040  Eval Sharpe:-0.200  Mean Ret:-23.29 bp
VALID eval: {'mean': -0.0014372060069137087, 'std': 0.030351988519612443, 'sharpe': -0.7516785283319689, 'steps': 3227}
TEST  eval: {'mean': -0.0015725876134731528, 'std': 0.08970929951150171, 'sharpe': -0.27827721960377716, 'steps': 7613}


In [8]:
# --- 8) Period Sharpe via deterministic rollouts per split ---
from rl_agent.trainer import deterministic_rewards, ann_sharpe

r_tr = deterministic_rewards(env_train, policy)
r_va = deterministic_rewards(env_valid, policy)
r_te = deterministic_rewards(env_test,  policy)

print("Sharpe train:", f"{ann_sharpe(r_tr):.3f}")
print("Sharpe valid:", f"{ann_sharpe(r_va):.3f}")
print("Sharpe test :", f"{ann_sharpe(r_te):.3f}")


Sharpe train: -0.200
Sharpe valid: -0.752
Sharpe test : -0.278


In [9]:
# --- 3b) OPTIONAL micro-fill (≤2 business days) for state_cols ---

# Work on a DatetimeIndex for time-weighted interpolation
panel = panel.sort_values("date").reset_index(drop=True)
panel_idx = panel.set_index("date")  # <- DatetimeIndex

# Ensure numeric dtype (avoid object columns blocking interpolate)
panel_idx[state_cols] = panel_idx[state_cols].apply(pd.to_numeric, errors="coerce")

print("NaN-window rate (pre):",
      panel_idx[state_cols].isna().rolling(60, min_periods=60)
      .apply(lambda s: s.isna().any(), raw=False).fillna(1).mean())

# Time-weighted interpolation limited to tiny gaps; only inside (no edge extrapolation)
panel_idx[state_cols] = panel_idx[state_cols].interpolate(
    method="time", limit=2, limit_direction="both", limit_area="inside"
)

print("NaN-window rate (post):",
      panel_idx[state_cols].isna().rolling(60, min_periods=60)
      .apply(lambda s: s.isna().any(), raw=False).fillna(1).mean())

# Restore the regular index
panel = panel_idx.reset_index()

# Sanity: returns untouched; daily scale looks reasonable?
print("ret_fwd daily std:", panel["ret_fwd"].std(ddof=1))


NaN-window rate (pre): iv_atm_30d_spx     0.003275
iv_ts_slope_spx    0.003275
iv_skew_30d_spx    0.003275
vix                0.003275
rate_10y           0.003275
rv_21d             0.003275
hvol_30d           0.003275
hvol_91d           0.003275
ret_fwd            0.003275
close_spy          0.003275
dtype: float64
NaN-window rate (post): iv_atm_30d_spx     0.003275
iv_ts_slope_spx    0.003275
iv_skew_30d_spx    0.003275
vix                0.003275
rate_10y           0.003275
rv_21d             0.003275
hvol_30d           0.003275
hvol_91d           0.003275
ret_fwd            0.003275
close_spy          0.003275
dtype: float64
ret_fwd daily std: 0.005513946410391123


# Gae Training for all 

### Actor–Critic (GAE) script run

We replace REINFORCE with a lower-variance Actor–Critic (policy + value head, GAE),
entropy/LR schedules, grad clipping, and checkpoint on best VALID Sharpe.


In [10]:
# --- GAE imports (use your scripts) ---
from rl_agent.train_ac_gae import train as train_gae
from rl_agent.experiment import build_envs, deterministic_rewards
from rl_agent.trainer import evaluate_env

from pathlib import Path
import numpy as np
import json


In [11]:
# --- Paths for artifacts ---
RUN_DIR = ROOT / "models" / "gae_run1"
CKPT    = RUN_DIR / "best.pt"
CONF    = RUN_DIR / "config.json"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# --- Choose window/hparams (start conservative; you can sweep later) ---
WINDOW = 45
LR     = 1e-2
ENT    = 0.15
STEPS  = 8000

# --- Train Actor–Critic (GAE) on your current panel & features ---
policy_ac, (tr_ac, va_ac, te_ac) = train_gae(
    panel=panel,                 # uses the same panel you've prepared above
    state_cols=state_cols,       # same features
    train_end="2017-12-31",
    valid_end="2019-12-31",
    window=WINDOW,
    txn_cost_bps=1.0,
    pos_limit=2.0,
    hidden=64,
    lr=LR,
    steps=STEPS,
    entropy_start=ENT,
    device="cpu",
    seed=42,
    save_ckpt=str(CKPT),         # saves best VALID Sharpe checkpoint
    save_config=str(CONF),       # saves run config JSON
    save_outdir=str(RUN_DIR),    # saves results.json + *_bps.csv + versions.json
)

print("GAE (VALID):", va_ac)
print("GAE (TEST) :", te_ac)


Env(train) ready. input_dim=450 (window=45, n_features=10)


KeyboardInterrupt: 

### I HAVENT RAN THE CODE BELOW AS THE CELL ABOVE TAKES FOREVER TO RUN

In [ ]:
# --- Build envs exactly as train sees them ---
envs = build_envs(
    panel=panel,
    features=state_cols,
    train_end="2017-12-31",
    valid_end="2019-12-31",
    window=WINDOW,
    txn_cost_bps=1.0,
    pos_limit=2.0,
)
env_tr, env_va, env_te = envs["env_tr"], envs["env_va"], envs["env_te"]

# --- Evaluate Sharpe deterministically ---
res_tr = evaluate_env(env_tr, policy_ac, deterministic=True)
res_va = evaluate_env(env_va, policy_ac, deterministic=True)
res_te = evaluate_env(env_te, policy_ac, deterministic=True)

print("Deterministic Sharpe — TRAIN:", f"{res_tr['sharpe']:.3f}")
print("Deterministic Sharpe — VALID:", f"{res_va['sharpe']:.3f}")
print("Deterministic Sharpe — TEST :", f"{res_te['sharpe']:.3f}")


In [ ]:
def ann_sharpe(bps):
    bps = np.asarray(bps, float)
    sd  = bps.std(ddof=1)
    return (bps.mean() / sd * np.sqrt(252)) if sd > 0 else 0.0

r_tr = deterministic_rewards(env_tr, policy_ac)
r_va = deterministic_rewards(env_va, policy_ac)
r_te = deterministic_rewards(env_te, policy_ac)

print("Curves Sharpe — TRAIN:", f"{ann_sharpe(r_tr):.3f}")
print("Curves Sharpe — VALID:", f"{ann_sharpe(r_va):.3f}")
print("Curves Sharpe — TEST :", f"{ann_sharpe(r_te):.3f}")
